# nimm_kalman 算法使用教程

本 notebook 面向算法使用者，说明 `nimm_kalman` 的用途、目录结构、公开插件和典型调用流程。

`nimm_kalman` 主要包含两类能力：

- Kalman 误差更新：根据最新预报和观测更新平均误差场或平均绝对误差场。
- Kalman 预报订正：使用已有误差场对新预报场进行订正。

公开插件类：

- `KalmanME`：更新 Kalman ME/MAE 误差场。
- `KalmanFix`：使用误差场订正预报。

真实业务运行依赖 `meteva_base`、标准六维格点数据和真实 nc 文件路径。本教程默认只执行轻量数值示例，不自动读写业务文件。

## 0. 打开和运行环境

建议使用 `pytorch` 环境作为 notebook kernel。

如果在 PyCharm Community 中只能预览 notebook，可以在命令行运行：

```powershell
conda activate pytorch
cd D:/nimm-file/cli_code/nimm_kalman/nbs
jupyter lab
```

然后在浏览器中打开 `kalman_tutorial.ipynb`。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "nbs":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(r"D:/nimm-file/cli_code/nimm_kalman")

WORKSPACE_ROOT = PROJECT_ROOT.parent
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

print("算法目录:", PROJECT_ROOT)
print("工作区目录:", WORKSPACE_ROOT)

## 1. 目录结构

| 目录/文件 | 说明 |
| --- | --- |
| `src/kalman_me_plugin.py` | `KalmanME` 插件，更新误差场 |
| `src/kalman_fix_plugin.py` | `KalmanFix` 插件，订正预报 |
| `src/kalman_cli.py` | 业务流程入口，按路径模板批量处理 nc |
| `src/data_transfer.py` | 将 ECMWF 土壤变量复制到 Kalman 处理目录 |
| `cli/kalman_data.py` | Kalman 业务处理命令行入口 |
| `cli/trans_data.py` | 资料拷贝命令行入口 |
| `test/` | 轻量数值单元测试 |

In [ ]:
for path in [
    PROJECT_ROOT / "src" / "kalman_me_plugin.py",
    PROJECT_ROOT / "src" / "kalman_fix_plugin.py",
    PROJECT_ROOT / "src" / "kalman_cli.py",
    PROJECT_ROOT / "src" / "data_transfer.py",
]:
    print(path)

## 2. 公开插件

插件统一从 `nimm_kalman.src` 导入：

```python
from nimm_kalman.src import KalmanME, KalmanFix
```

`KalmanME.process(fcst_new, obs_new, me_before=None)`：

- `fcst_new`：最新预报格点。
- `obs_new`：对应观测格点。
- `me_before`：上一时刻误差场，可为空。
- 返回更新后的误差场。

`KalmanFix.process(fcst_new, me_before=None)`：

- `fcst_new`：待订正预报。
- `me_before`：误差场。
- 返回订正后的预报。

In [ ]:
from nimm_kalman.src import KalmanME, KalmanFix

me_plugin = KalmanME(alpha=0.1, is_mae=False)
fix_plugin = KalmanFix()

print(me_plugin)
print(fix_plugin)

## 3. 轻量数值示例

下面用数组演示 Kalman 的两个核心数值动作：

1. 计算预报和观测差值。
2. 用衰减系数 `alpha` 更新误差场。
3. 用误差场订正预报。

这个示例不依赖真实 `meteva_base` 格点对象，适合快速理解算法含义。

In [ ]:
import numpy as np

from nimm_kalman.utils.grid_utils import (
    decaying_fcst,
    decaying_me,
    forecast_observation_difference,
)

fcst = np.array([3.0, 5.0], dtype=np.float32)
obs = np.array([1.0, 7.0], dtype=np.float32)
previous_me = np.array([1.0, 3.0], dtype=np.float32)

latest_error = forecast_observation_difference(fcst, obs, absolute=False)
updated_me = decaying_me(latest_error, previous_me, alpha=0.25)
corrected_fcst = decaying_fcst(fcst, updated_me)

print("最新误差:", latest_error)
print("更新后 ME:", updated_me)
print("订正后预报:", corrected_fcst)

## 4. 真实业务流程

真实业务入口在 `nimm_kalman.src.kalman_cli.process`，它会按路径模板读取预报、观测和历史误差场，并可选输出订正结果。

核心参数：

| 参数 | 说明 |
| --- | --- |
| `fcst_path` | 预报 nc 路径模板 |
| `obs_path` | 观测 nc 路径模板 |
| `kalme_path` | Kalman ME 文件路径模板 |
| `time` | 起报时间，`datetime` 对象 |
| `dtimes` | 预报时效列表 |
| `alpha` | 误差场衰减更新系数 |
| `is_mae` | 是否更新平均绝对误差 |
| `output` | 订正结果输出路径模板 |
| `is_kal_fix` | 是否执行预报订正 |

路径模板需要能被 `meteva_base.get_path()` 识别。

In [ ]:
# 真实业务调用模板：确认路径和数据存在后再取消注释执行。
# from datetime import datetime
# from nimm_kalman.src.kalman_cli import process
#
# kal_me_list, result_list = process(
#     fcst_path="/path/to/fcst/YYYYMMDDHH.TTT.nc",
#     obs_path="/path/to/obs/YYYYMMDDHH.nc",
#     time=datetime(2026, 1, 1, 8),
#     dtimes=[24, 48, 72],
#     kalme_path="/path/to/kalman_me/YYYYMMDDHH.TTT.nc",
#     alpha=0.1,
#     is_mae=False,
#     output="/path/to/output/YYYYMMDDHH.TTT.nc",
#     keep_in_memory=True,
# )

## 5. 资料拷贝流程

`nimm_kalman.src.data_transfer.copy_process_data` 用于把 ECMWF 土壤变量复制到 Kalman 处理目录。

默认变量：

- `SWVL`：土壤湿度。
- `STL`：土壤温度。

默认层级映射：

| 原始层级 | 目标层级 |
| --- | --- |
| `0-7` | `5` |
| `7-28` | `10` |
| `28-100` | `40` |
| `100-MISSING` | `100` |

真实运行前请确认源目录和目标目录。

In [ ]:
# 资料拷贝模板：确认路径后再取消注释执行。
# from datetime import datetime
# from nimm_kalman.src.data_transfer import copy_process_data
#
# copied = copy_process_data(run_date=datetime(2026, 1, 1))
# print("复制文件数:", copied)

## 6. 命令行等价调用

资料拷贝：

```powershell
cd D:/nimm-file/cli_code
python -m nimm_kalman.cli.trans_data
```

Kalman 业务处理：

```powershell
cd D:/nimm-file/cli_code
python -m nimm_kalman.cli.kalman_data --help
```

具体参数以 `--help` 输出为准。

## 7. 使用前检查清单

- `pytorch` 环境中可以导入 `meteva_base`。
- 预报、观测、ME 文件均为标准六维格点结构。
- 预报和观测经纬度网格一致或可插值到同一网格。
- `fcst_path`、`obs_path`、`kalme_path`、`output` 路径模板正确。
- 首次运行建议只处理少量时效，确认输出无误后再批量运行。
- `alpha` 越大，最新误差对 ME 更新影响越强。